# 1. 선형 변환과 편향(bias)의 역할

선형 회귀(Linear Regression)의 핵심은 **데이터에 가장 잘 맞는 직선(또는 평면)** 을 찾는 것입니다.  
이때 모델은 아래 식으로 표현됩니다.

$$
\hat{y} = XW + b
$$

- $X$: 입력 데이터(특징)
- $W$: 가중치(각 특징의 중요도)
- $b$: 편향(절편, 기준점 이동)
- $\hat{y}$: 예측값

---

## 왜 bias가 꼭 필요할까?

Stage 1에서 주로 본 것은 $XW$였습니다.  
하지만 $XW$만 사용하면 모델은 다음 제약을 갖습니다.

- 입력이 0이면 출력도 0
- 결정 경계(또는 회귀선)가 원점을 지나야 함

즉, 모델의 표현력이 제한됩니다.  
여기에 $b$를 더하면 직선 전체를 위/아래로 이동시킬 수 있어 데이터에 훨씬 유연하게 맞출 수 있습니다.

---

## 직관으로 이해하기

### 1) 가중치 $W$의 역할

- 직선의 **기울기(slope)** 를 결정
- 입력이 1 증가할 때 출력이 얼마나 바뀌는지 조절

### 2) 편향 $b$의 역할

- 직선의 **위치(절편, intercept)** 를 결정
- 기울기는 그대로 두고 그래프를 평행 이동

예시:

- 모델 A: $\hat{y}=2x$
- 모델 B: $\hat{y}=2x+3$

두 모델은 기울기(2)는 같지만, B는 전체가 위로 3만큼 이동합니다.

---

## 분류 모델에서도 bias가 중요한 이유

편향은 결정 경계의 위치를 이동시킵니다.

- bias가 없으면 경계가 원점을 지나야 함
- bias가 있으면 데이터 분포에 맞게 경계를 자유롭게 이동 가능

그래서 분류/회귀 모두에서 bias는 성능에 큰 영향을 줍니다.

---

## 학습 관점에서 보는 $W$와 $b$

선형 회귀는 손실 함수(예: MSE)를 최소화하도록 $W, b$를 함께 업데이트합니다.

$$
\text{MSE} = \frac{1}{N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2
$$

- $W$ 업데이트: 기울기를 조절해 데이터의 변화율을 맞춤
- $b$ 업데이트: 예측 전체를 위/아래로 이동해 평균 오차를 줄임

즉, **좋은 선형 회귀 모델은 $W$와 $b$를 동시에 잘 찾은 모델**입니다.

---

## 실전 체크포인트

- 학습 후 $b$가 0 근처라고 항상 좋은 것은 아님
- 데이터 스케일링을 하면 $W, b$의 해석이 더 안정적일 수 있음
- 과소적합이면 특징 부족/선형 가정 한계 가능성 점검
- 과적합이면 정규화(예: Ridge/Lasso) 고려

---

## 핵심 한 줄 정리

선형 회귀의 목적은 **데이터에 가장 잘 맞는 직선을 찾는 것**이며,  
그 직선을 결정하는 핵심 파라미터가 바로 **$W$(기울기)와 $b$(절편)** 입니다.

## 1. 데이터 준비 

In [2]:
import torch
import torch.optim as optim
x_train = torch.tensor([[1], [2], [3]], dtype=torch.float)
y_train = torch.tensor([[3],[6],[9]], dtype=torch.float)

print(x_train)
print(y_train)

tensor([[1.],
        [2.],
        [3.]])
tensor([[3.],
        [6.],
        [9.]])


## 가중치와 편향을 시작하는 다양한 방법



In [9]:
W = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
# requires_grad=True를 설정하면, 이 텐서에 대한 연산을 추적하고, 역전파 시 미분값을 계산할 수 있다.

print(f"가중치:",W)
print(f"편향:",b)

pred = x_train * W + b
loss_ = torch.mean((pred - y_train) ** 2) #MSE 손실함수

#역전파 과정 
loss_.backward() #loss_를 W와 b에 대해 미분
print(f"가중치 W에 대한 미분값:", W.grad)
print(f"편향 b에 대한 미분값:", b.grad)

#SGD, Stochastic Gradient Descent 확률적 경사 하강법
optimizer = optim.SGD([W, b], lr=0.01) #학습, lr = learning rate
# 학습 과정에서 W와 b의 값을 자동으로 조정한다. optimizer.step()을 호출하면, W와 b의 값을 업데이트한다.



가중치: tensor([0.], requires_grad=True)
편향: tensor([0.], requires_grad=True)
가중치 W에 대한 미분값: tensor([-28.])
편향 b에 대한 미분값: tensor([-12.])


In [11]:
display(x_train)
display(y_train)
# 전체 에폭 수를 정의합니다.
epochs = 1000

for epoch in range(epochs):
    # 순전파: 모델을 통해 예측값을 계산합니다.
    pred = x_train * W + b
    
    # 손실 함수를 계산합니다.
    loss = torch.mean((pred - y_train) ** 2)
    
    # 역전파: 손실 함수의 기울기(gradient)를 계산합니다.
    optimizer.zero_grad()  # 기울기를 0으로 초기화
    loss.backward()  # 역전파 수행
    
    # 옵티마이저를 사용하여 파라미터를 업데이트합니다.
    optimizer.step()
    
    # 일정 에폭마다 학습 상태를 출력합니다.
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], W: {W.item():.3f}, b: {b.item():.3f}, Loss: {loss.item():.4f}')

tensor([[1.],
        [2.],
        [3.]])

tensor([[3.],
        [6.],
        [9.]])

Epoch [100/1000], W: 2.966, b: 0.078, Loss: 0.0009
Epoch [200/1000], W: 2.973, b: 0.062, Loss: 0.0005
Epoch [300/1000], W: 2.979, b: 0.048, Loss: 0.0003
Epoch [400/1000], W: 2.983, b: 0.038, Loss: 0.0002
Epoch [500/1000], W: 2.987, b: 0.030, Loss: 0.0001
Epoch [600/1000], W: 2.990, b: 0.024, Loss: 0.0001
Epoch [700/1000], W: 2.992, b: 0.018, Loss: 0.0000
Epoch [800/1000], W: 2.994, b: 0.015, Loss: 0.0000
Epoch [900/1000], W: 2.995, b: 0.011, Loss: 0.0000
Epoch [1000/1000], W: 2.996, b: 0.009, Loss: 0.0000


In [13]:
test_x = torch.tensor([[10]], dtype=torch.float)

with torch.no_grad(): #자동 미분 기능을 비활성하도록 지시, 이는 추론(inference) 또는 예측 수행시에는 모델 파라미터의 기울기를 계산할 필요가 없기 때문
    pred_y = test_x * W + b
    print("훈련 후 입력이 10일 때의 예측값 :", pred_y.item()) #y = 3x의 관계를 모델이 학습했음을 시사 

훈련 후 입력이 10일 때의 예측값 : 29.969486236572266


# nn.Modulerhk nn.Linear를 활용한 단순 선형 회귀 모델 구현 



In [16]:
#데이터 준비 
import torch 
import torch.nn as nn
torch.manual_seed(42) # 난수 코드 실행간 난수의 일관된 결과를 얻기 위해 

x_train = torch.tensor([[1], [2], [3]], dtype=torch.float)
y_train = torch.tensor([[3], [6], [9]], dtype=torch.float)

# 입력 및 출력 특성의 크기 계산
in_features = x_train.shape[1]
out_features = y_train.shape[1]

print(in_features)
print(out_features)

# 선형 회귀 모델 초기화
model = nn.Linear(in_features=1, out_features=1) #선형 회귀 모델을 초기화 
#nn.Linear는 torch.nn모듈에 포함된 클래스, 선형변환을 적용하는 레이어를 나타냄 
#입력 특성크기(in_features)와 출력 특성 크기(out_features)를 인자로 받아, 해당 크기에 맞는 가중치와 편향을 내부적으로 초기화 

1
1


- in_feature는 입력 특성의 크기, x_train.shape[1]을 통해 계산, x_train 텐서의 열의 수(입력 데이터 특성 수)를 의미 
- out_feature는 모델의 출력 특성의 크기를 나타냄. y_train.shape[1]을 통해 계산 되며, y_train 텐서의 열의 수( 출력 데이터의 특성 수 )를 의미

In [19]:
#모델의 파라미터 조회를 통한 초기 가중치와 편향의 자동 초기화 이해 
for param in model.parameters():
    print(param.data)
#PyTorch의 nn.Linear는 내부적으로 가중치와 편향을 초기화하며, 이 초기화는 일반적으로 작은 랜덤 값으로 이루어집니다.

tensor([[0.7645]])
tensor([0.8300])


## 실행 결과가 조금 다르게 나오나요?

같은 코드를 실행했는데도 책이나 강의 화면과 **숫자가 조금 다르게 나오는 경우는 매우 흔합니다.**  
`torch.manual_seed(42)`처럼 시드를 고정해도, 환경이 완전히 같지 않다면 결과가 조금 달라질 수 있습니다.

---

### 왜 이런 일이 생길까?

#### 1. 시드 고정은 "완전한 동일성"이 아니라 "재현 가능성을 높이는 설정"

`torch.manual_seed()`는 난수의 시작점을 맞춰 주지만, 아래 요소까지 완전히 같게 만들지는 못합니다.

- CPU와 GPU의 연산 방식 차이
- 라이브러리 내부 최적화 방식 차이
- 병렬 연산 순서 차이
- 연산 장치 및 드라이버 차이

즉, **시드 고정은 결과를 비슷하게 만드는 장치이지, 모든 환경에서 100% 동일한 값을 보장하는 장치는 아닙니다.**

---

#### 2. 실행 환경이 다르면 결과도 달라질 수 있음

다음 요소가 다르면 같은 코드라도 결과가 조금씩 달라질 수 있습니다.

- PyTorch 버전
- Python 버전
- CUDA 버전
- 운영체제 차이
- 사용 중인 하드웨어 차이

특히 딥러닝 프레임워크는 내부적으로 속도 최적화를 많이 수행하기 때문에, **보이지 않는 내부 계산 순서 차이**가 결과에 영향을 줄 수 있습니다.

---

### 그래서 결과가 다르면 잘못된 걸까?

그렇지는 않습니다.  
중요한 것은 **숫자가 완전히 똑같은지**보다 아래가 맞는지입니다.

- 학습이 정상적으로 진행되는가?
- 손실(loss)이 점점 감소하는가?
- 가중치와 편향이 합리적인 방향으로 업데이트되는가?
- 예측값이 정답에 가까워지는가?

즉, **학습 흐름과 방향이 맞다면 작은 수치 차이는 자연스러운 현상**입니다.

---

### 실전에서 이렇게 확인하세요

#### 체크 1. 시드 고정

```python
import torch

torch.manual_seed(42)
```

#### 체크 2. 같은 라이브러리 버전인지 확인

```python
import torch
print(torch.__version__)
```

#### 체크 3. CPU/GPU 환경 차이 인지하기

- CPU에서 실행한 결과와 GPU에서 실행한 결과는 조금 다를 수 있음
- 특히 소수점 단위의 미세한 차이는 흔함

---

### 핵심 정리

- 시드를 고정해도 결과가 조금 다를 수 있습니다.
- 이는 보통 **환경 차이와 내부 연산 방식 차이** 때문입니다.
- 따라서 숫자가 조금 다르더라도 **학습 과정의 방향과 결과 해석**이 맞으면 괜찮습니다.

> 숫자 하나하나보다, 모델이 올바르게 학습되고 있는지를 먼저 보세요.